In [ ]:
import os
import sys
import warnings
import subprocess
import pkgutil

warnings.filterwarnings("ignore")

for package in ["shap", "statsmodels"]:
    if pkgutil.find_loader(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

import shap
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import MNLogit
from statsmodels.stats.outliers_influence import variance_inflation_factor

RANDOM_STATE = 42

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 100)
plt.rcParams["figure.figsize"] = (8, 5)

In [ ]:
DATA_PATH = "/content/diabetes_012_health_indicators_BRFSS2015.csv"

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Dataset not found. Upload diabetes_012_health_indicators_BRFSS2015.csv and update DATA_PATH if needed.")

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
display(df.head())

In [ ]:
print("Shape before cleaning:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nMissing values:")
display(df.isna().sum().to_frame("Missing Values"))
print("\nDuplicated rows:", df.duplicated().sum())

In [ ]:
df = df.copy()
df.columns = df.columns.str.strip()

if "Diabetes_012" not in df.columns:
    raise ValueError("Target column Diabetes_012 was not found.")

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

duplicate_rows_before = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)

binary_cols = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost",
    "DiffWalk", "Sex"
]

for col in binary_cols:
    if col in df.columns:
        df.loc[~df[col].isin([0, 1]), col] = np.nan

range_rules = {
    "Diabetes_012": (0, 2),
    "BMI": (10, 100),
    "GenHlth": (1, 5),
    "MentHlth": (0, 30),
    "PhysHlth": (0, 30),
    "Age": (1, 13),
    "Education": (1, 6),
    "Income": (1, 8),
}

for col, (low, high) in range_rules.items():
    if col in df.columns:
        df.loc[~df[col].between(low, high), col] = np.nan

df = df.dropna(subset=["Diabetes_012"]).reset_index(drop=True)
df["Diabetes_012"] = df["Diabetes_012"].astype(int)

print("Duplicate rows removed:", duplicate_rows_before)
print("Shape after cleaning:", df.shape)
display(df.head())

In [ ]:
expected_features = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income"
]

features = [col for col in expected_features if col in df.columns]
target = "Diabetes_012"

X = df[features].copy()
y = df[target].copy().astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Features used:", features)

In [ ]:
print(y.value_counts().sort_index())
print("\nClass percentages:")
display((y.value_counts(normalize=True).sort_index() * 100).round(2).to_frame("Percentage"))

In [ ]:
display(X.describe().T)

In [ ]:
class_counts = y.value_counts().sort_index()
class_percent = (class_counts / class_counts.sum() * 100).round(2)

ax = class_counts.plot(kind="bar", title="Distribution of Diabetes_012")
ax.set_xlabel("Class")
ax.set_ylabel("Count")
plt.xticks(rotation=0)
plt.show()

display(pd.DataFrame({"Count": class_counts, "Percentage": class_percent}))

In [ ]:
selected_distribution_features = [col for col in ["BMI", "MentHlth", "PhysHlth", "GenHlth", "Age", "Income"] if col in X.columns]

if len(selected_distribution_features) > 0:
    X[selected_distribution_features].hist(figsize=(12, 8), bins=20)
    plt.suptitle("Distribution of Selected Features")
    plt.show()

In [ ]:
if "BMI" in df.columns:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df[target], y=df["BMI"])
    plt.title("BMI by Diabetes_012")
    plt.xlabel("Diabetes_012")
    plt.ylabel("BMI")
    plt.show()

In [ ]:
plot_features = [col for col in ["HighBP", "HighChol", "PhysActivity"] if col in df.columns]

if len(plot_features) > 0:
    fig, axes = plt.subplots(1, len(plot_features), figsize=(6 * len(plot_features), 5))

    if len(plot_features) == 1:
        axes = [axes]

    for ax, feature in zip(axes, plot_features):
        sns.countplot(x=feature, hue=target, data=df, ax=ax)
        ax.set_title(f"{feature} vs Diabetes_012")

    plt.tight_layout()
    plt.show()

In [ ]:
corr_matrix = df[features + [target]].corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, cmap="RdYlGn", annot=False)
plt.title("Correlation Heatmap")
plt.show()

correlation_with_target = corr_matrix[target].drop(target, errors="ignore")
correlation_with_target = correlation_with_target.reindex(
    correlation_with_target.abs().sort_values(ascending=False).index
)

display(correlation_with_target.to_frame("Correlation with Diabetes_012"))

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print("X_train_raw shape:", X_train_raw.shape)
print("X_test_raw shape:", X_test_raw.shape)

In [ ]:
imputer = SimpleImputer(strategy="median")

X_train_imputed = pd.DataFrame(
    imputer.fit_transform(X_train_raw),
    columns=X_train_raw.columns,
    index=X_train_raw.index
)

X_test_imputed = pd.DataFrame(
    imputer.transform(X_test_raw),
    columns=X_test_raw.columns,
    index=X_test_raw.index
)

print("Missing values in X_train:", X_train_imputed.isnull().sum().sum())
print("Missing values in X_test:", X_test_imputed.isnull().sum().sum())

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=X_train_imputed.columns,
    index=X_train_imputed.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=X_test_imputed.columns,
    index=X_test_imputed.index
)

print("Scaled training shape:", X_train_scaled_df.shape)
print("Scaled testing shape:", X_test_scaled_df.shape)

In [ ]:
k = min(15, X_train_scaled_df.shape[1])

anova_selector = SelectKBest(score_func=f_classif, k=k)

X_train_selected = anova_selector.fit_transform(X_train_scaled_df, y_train)
X_test_selected = anova_selector.transform(X_test_scaled_df)

selected_mask = anova_selector.get_support()
selected_features = X_train_scaled_df.columns[selected_mask].tolist()

X_train_selected_df = pd.DataFrame(
    X_train_selected,
    columns=selected_features,
    index=X_train_scaled_df.index
)

X_test_selected_df = pd.DataFrame(
    X_test_selected,
    columns=selected_features,
    index=X_test_scaled_df.index
)

print("Shape after feature selection:", X_train_selected_df.shape)
print("Selected features:")
print(selected_features)

In [ ]:
anova_scores = pd.DataFrame({
    "Feature": X_train_scaled_df.columns,
    "F_score": anova_selector.scores_,
    "p_value": anova_selector.pvalues_
}).sort_values(by="F_score", ascending=False)

display(anova_scores)

plt.figure(figsize=(10, 6))
sns.barplot(data=anova_scores, x="F_score", y="Feature")
plt.title("ANOVA Feature Importance")
plt.xlabel("F-score")
plt.ylabel("Feature")
plt.show()

In [ ]:
log_reg = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    solver="lbfgs",
    random_state=RANDOM_STATE
)

log_reg.fit(X_train_selected_df, y_train)

print("Model trained successfully.")

In [ ]:
y_pred = log_reg.predict(X_test_selected_df)
y_prob = log_reg.predict_proba(X_test_selected_df)

print("Prediction shape:", y_pred.shape)
print("Probability shape:", y_prob.shape)

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob, multi_class="ovr", average="weighted")

metrics_df = pd.DataFrame([{
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    "ROC-AUC": roc_auc
}])

display(metrics_df.round(4))

In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
classes = sorted(y.unique().tolist())

cm = confusion_matrix(y_test, y_pred, labels=classes)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {cls}" for cls in classes],
    columns=[f"Pred {cls}" for cls in classes]
)

display(cm_df)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
y_test_bin = label_binarize(y_test, classes=classes)

plt.figure(figsize=(8, 6))

for i, cls in enumerate(classes):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    roc_auc_cls = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"Class {cls} vs Rest (AUC = {roc_auc_cls:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Multiclass ROC Curve")
plt.legend()
plt.show()

In [ ]:
coef_df = pd.DataFrame(
    log_reg.coef_.T,
    index=selected_features,
    columns=[f"Class_{cls}" for cls in log_reg.classes_]
)

display(coef_df)

In [ ]:
for cls in log_reg.classes_:
    top_features = coef_df[f"Class_{cls}"].sort_values(ascending=False).head(10)

    plt.figure(figsize=(8, 5))
    plt.barh(top_features.index, top_features.values)
    plt.gca().invert_yaxis()
    plt.title(f"Top Positive Coefficients for Class {cls}")
    plt.xlabel("Coefficient")
    plt.show()

In [ ]:
sample_size = min(500, len(X_test_selected_df))
X_sample = X_test_selected_df.iloc[:sample_size]

print("Sample shape:", X_sample.shape)

In [ ]:
explainer = shap.Explainer(log_reg, X_train_selected_df)
shap_values = explainer(X_sample)

for class_index, cls in enumerate(log_reg.classes_):
    shap.plots.beeswarm(shap_values[:, :, class_index], show=False)
    plt.title(f"SHAP Beeswarm for Class {cls}")
    plt.show()

In [ ]:
bt_df = X_train_imputed.copy()
bt_df["Diabetes_012"] = y_train.values

continuous_vars = [col for col in ["BMI", "MentHlth", "PhysHlth"] if col in bt_df.columns]

for col in continuous_vars:
    bt_df = bt_df[bt_df[col] > 0]

for col in continuous_vars:
    bt_df[f"{col}_log"] = bt_df[col] * np.log(bt_df[col])

display(bt_df.head())

In [ ]:
if len(continuous_vars) > 0:
    X_bt = bt_df[continuous_vars + [f"{col}_log" for col in continuous_vars]]
    X_bt = sm.add_constant(X_bt)
    y_bt = bt_df["Diabetes_012"]

    bt_model = MNLogit(y_bt, X_bt).fit(method="newton", disp=False)
    print(bt_model.summary())
else:
    print("No valid continuous variables found for this check.")

In [ ]:
sample_size = min(5000, len(X_train_selected_df))
rng = np.random.RandomState(RANDOM_STATE)
sample_idx = rng.choice(len(X_train_selected_df), size=sample_size, replace=False)

X_sample_outlier = X_train_selected_df.iloc[sample_idx]
y_sample = y_train.iloc[sample_idx]

log_reg_sample = LogisticRegression(
    max_iter=3000,
    class_weight="balanced",
    solver="lbfgs",
    random_state=RANDOM_STATE
)

log_reg_sample.fit(X_sample_outlier, y_sample)

y_sample_pred = log_reg_sample.predict(X_sample_outlier)
y_sample_prob = log_reg_sample.predict_proba(X_sample_outlier)

print("Sample model trained successfully.")

In [ ]:
max_prob = y_sample_prob.max(axis=1)
misclassified = y_sample_pred != y_sample.to_numpy()

outlier_df = pd.DataFrame({
    "Actual": y_sample.to_numpy(),
    "Predicted": y_sample_pred,
    "Max_Prob": max_prob,
    "Misclassified": misclassified
})

potential_outliers = outlier_df[(outlier_df["Misclassified"] == True) & (outlier_df["Max_Prob"] > 0.90)]
uncertain_cases = outlier_df[outlier_df["Max_Prob"] < 0.40]

print("Potential influential / problematic cases:", len(potential_outliers))
print("Highly uncertain cases:", len(uncertain_cases))

display(outlier_df.head())

In [ ]:
plt.figure(figsize=(7, 4))
plt.hist(outlier_df["Max_Prob"], bins=30)
plt.title("Distribution of Maximum Predicted Probability")
plt.xlabel("Maximum Predicted Probability")
plt.ylabel("Frequency")
plt.show()

In [ ]:
vif_sample_size = min(5000, len(X_train_selected_df))
X_vif = X_train_selected_df.sample(n=vif_sample_size, random_state=RANDOM_STATE)

X_vif_const = sm.add_constant(X_vif)

vif_data = pd.DataFrame()
vif_data["Feature"] = X_vif_const.columns
vif_data["VIF"] = [
    variance_inflation_factor(X_vif_const.values, i)
    for i in range(X_vif_const.shape[1])
]

display(vif_data.sort_values("VIF", ascending=False))

In [ ]:
print("Duplicate rows removed during preprocessing:", duplicate_rows_before)
print("Duplicate rows after preprocessing:", df.duplicated().sum())
print("Shape after duplicate removal:", df.shape)

In [ ]:
class_counts = y.value_counts().sort_index()

print("Class counts:")
print(class_counts)

print("\nMinimum class count:", class_counts.min())
print("Total sample size:", len(y))